# E0 v3 — signal, permutation, and random-k baseline

One Kaggle session runs all five LUAD-versus-LUSC cohorts. The protocol uses F-test + LR-L2 at `k=10`, 200 label permutations, and 100 random-gene draws. Every permutation/draw is a separate checkpoint, so re-running resumes safely.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
COHORTS = ['tcga_luad_lusc_primary_tumor', 'gse10245', 'gse30219', 'gse37745', 'gse42127']
K = 10
N_PERMUTATIONS = 200
N_RANDOM_DRAWS = 100
SEED = 20261027
PREFILTER_P0 = 5000

REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
# Do not install requirements.txt on Kaggle: its base scientific stack is already compatible.
RUNTIME_IMPORTS = ('numpy', 'pandas', 'scipy', 'sklearn', 'tables', 'joblib', 'threadpoolctl')
for package in RUNTIME_IMPORTS:
    print(package, __import__(package).__version__)

WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    source, target = INPUT_V3_DATASET / folder, WORK_ROOT / folder
    assert source.is_dir(), source
    if not target.exists():
        target.symlink_to(source, target_is_directory=True)


In [ ]:
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.data import load_dataset, sha256, variance_prefilter
from survey_v2.e1_splits import validate_repeated_split_manifest
from scripts.run_e1_feature_selection import split_indices

def atomic_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)

def fit_and_score(X_train, y_train, X_test, y_test, columns, seed):
    model = Pipeline([
        ('scale', StandardScaler()),
        ('lr', LogisticRegression(C=1.0, class_weight='balanced', max_iter=5000, random_state=seed)),
    ])
    with threadpool_limits(1):
        model.fit(X_train[:, columns], y_train)
        score = model.predict_proba(X_test[:, columns])[:, 1]
    pred = (score >= 0.5).astype(int)
    return float(balanced_accuracy_score(y_test, pred)), float(roc_auc_score(y_test, score))

def ftest_panel(X_train, y_train, prefilter):
    stat, _ = f_classif(X_train[:, prefilter], y_train)
    order = np.argsort(-np.nan_to_num(stat, nan=-np.inf), kind='stable')
    return prefilter[order[:K]]

def permute_by_patient(y, groups, rng):
    labels = pd.Series(y, index=groups).groupby(level=0).first()
    if (pd.Series(y, index=groups).groupby(level=0).nunique() == 1).all():
        mapping = dict(zip(labels.index, rng.permutation(labels.to_numpy())))
        return np.array([mapping[g] for g in groups], dtype=int)
    return rng.permutation(y)


In [ ]:
# Real signal, permutation null, and random-k baseline. Checkpoint granularity is one draw.
for COHORT in COHORTS:
    ds = load_dataset(WORK_ROOT, COHORT)
    manifest = pd.read_csv(ds.data_dir / 'repeated_splits.csv')
    errors = validate_repeated_split_manifest(manifest, n_folds=C.N_FOLDS)
    assert not errors, errors
    root = WORK_ROOT / 'results' / 'survey_v3' / 'E0_signal' / COHORT
    real_path = root / 'checkpoints' / 'real.csv'

    if not real_path.is_file():
        rows = []
        for repeat in range(len(C.REPEAT_SEEDS)):
            for fold in range(C.N_FOLDS):
                train, test = split_indices(ds, manifest, repeat, fold)
                seed = int(manifest[(manifest.repeat == repeat) & (manifest.fold == fold)].seed.iloc[0])
                pre = variance_prefilter(ds.X[train], PREFILTER_P0)
                panel = ftest_panel(ds.X[train], ds.y[train], pre)
                bacc, auroc = fit_and_score(ds.X[train], ds.y[train], ds.X[test], ds.y[test], panel, seed)
                rows.append({'cohort': COHORT, 'analysis': 'real', 'repeat': repeat, 'fold': fold, 'seed': seed, 'k': K, 'bacc': bacc, 'auroc': auroc})
        atomic_csv(pd.DataFrame(rows), real_path)

    for pid in range(N_PERMUTATIONS):
        path = root / 'checkpoints' / 'permutation' / f'pid_{pid:03d}.csv'
        if path.is_file():
            continue
        y_perm = permute_by_patient(ds.y, ds.groups, np.random.default_rng([SEED, 1, pid]))
        rows = []
        for repeat in range(len(C.REPEAT_SEEDS)):
            for fold in range(C.N_FOLDS):
                train, test = split_indices(ds, manifest, repeat, fold)
                seed = int(manifest[(manifest.repeat == repeat) & (manifest.fold == fold)].seed.iloc[0])
                pre = variance_prefilter(ds.X[train], PREFILTER_P0)
                panel = ftest_panel(ds.X[train], y_perm[train], pre)
                bacc, auroc = fit_and_score(ds.X[train], y_perm[train], ds.X[test], y_perm[test], panel, seed)
                rows.append({'cohort': COHORT, 'analysis': 'permutation', 'draw': pid, 'repeat': repeat, 'fold': fold, 'seed': seed, 'k': K, 'bacc': bacc, 'auroc': auroc})
        atomic_csv(pd.DataFrame(rows), path)
        print(f'{COHORT}: permutation {pid + 1}/{N_PERMUTATIONS}', flush=True)

    for draw in range(N_RANDOM_DRAWS):
        path = root / 'checkpoints' / 'random_k' / f'draw_{draw:03d}.csv'
        if path.is_file():
            continue
        rows = []
        for repeat in range(len(C.REPEAT_SEEDS)):
            for fold in range(C.N_FOLDS):
                train, test = split_indices(ds, manifest, repeat, fold)
                seed = int(manifest[(manifest.repeat == repeat) & (manifest.fold == fold)].seed.iloc[0])
                pre = variance_prefilter(ds.X[train], PREFILTER_P0)
                panel = np.random.default_rng([SEED, 2, draw, repeat, fold]).choice(pre, size=K, replace=False)
                bacc, auroc = fit_and_score(ds.X[train], ds.y[train], ds.X[test], ds.y[test], panel, seed)
                rows.append({'cohort': COHORT, 'analysis': 'random_k', 'draw': draw, 'repeat': repeat, 'fold': fold, 'seed': seed, 'k': K, 'bacc': bacc, 'auroc': auroc})
        atomic_csv(pd.DataFrame(rows), path)
        print(f'{COHORT}: random-k {draw + 1}/{N_RANDOM_DRAWS}', flush=True)


In [ ]:
# Consolidate only when all checkpoints exist, then create one Kaggle output ZIP.
all_summary = []
for COHORT in COHORTS:
    root = WORK_ROOT / 'results' / 'survey_v3' / 'E0_signal' / COHORT
    perm_paths = sorted((root / 'checkpoints' / 'permutation').glob('pid_*.csv'))
    random_paths = sorted((root / 'checkpoints' / 'random_k').glob('draw_*.csv'))
    assert len(perm_paths) == N_PERMUTATIONS, (COHORT, len(perm_paths), N_PERMUTATIONS)
    assert len(random_paths) == N_RANDOM_DRAWS, (COHORT, len(random_paths), N_RANDOM_DRAWS)
    real = pd.read_csv(root / 'checkpoints' / 'real.csv')
    permutation = pd.concat([pd.read_csv(path) for path in perm_paths], ignore_index=True)
    random_k = pd.concat([pd.read_csv(path) for path in random_paths], ignore_index=True)
    atomic_csv(real, root / 'e0_real_scores.csv')
    atomic_csv(permutation, root / 'e0_permutation_scores.csv')
    atomic_csv(random_k, root / 'e0_random_k_scores.csv')
    real_mean = real[['bacc', 'auroc']].mean()
    null = permutation.groupby('draw')[['bacc', 'auroc']].mean()
    random_mean = random_k.groupby('draw')[['bacc', 'auroc']].mean()
    row = {'cohort': COHORT, 'method': 'f_test_lr_l2', 'k': K, 'n_outer_splits': len(real), 'n_permutations': N_PERMUTATIONS, 'n_random_draws': N_RANDOM_DRAWS}
    for metric in ('bacc', 'auroc'):
        observed = float(real_mean[metric])
        row[f'{metric}_real_mean'] = observed
        row[f'{metric}_permutation_mean'] = float(null[metric].mean())
        row[f'{metric}_permutation_p_one_sided'] = float((1 + (null[metric] >= observed).sum()) / (N_PERMUTATIONS + 1))
        row[f'{metric}_random_k_mean'] = float(random_mean[metric].mean())
        row[f'{metric}_gain_over_random_k'] = observed - float(random_mean[metric].mean())
    summary = pd.DataFrame([row])
    atomic_csv(summary, root / 'e0_summary.csv')
    all_summary.append(summary)

result_root = WORK_ROOT / 'results' / 'survey_v3' / 'E0_signal'
atomic_csv(pd.concat(all_summary, ignore_index=True), result_root / 'e0_summary_all_cohorts.csv')
config = {'cohorts': COHORTS, 'method': 'f_test_lr_l2', 'k': K, 'n_permutations': N_PERMUTATIONS, 'n_random_draws': N_RANDOM_DRAWS, 'seed': SEED, 'prefilter_p0': PREFILTER_P0, 'protocol': '5x5 repeated outer CV; all fitting and F-test ranking are train-only'}
(result_root / 'e0_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
archive_name = '/kaggle/working/e0_signal_all_cohorts.zip'
archive = shutil.make_archive(archive_name.removesuffix('.zip'), 'zip', result_root)
print(archive)
